# Notebook 3 — Analysis & visualisation

Goal: answer the research question with statistics and visuals.

> **Research question:** Which structural and geographic factors most strongly explain rent differences in Switzerland — and how does the official rent index compare to current asking prices?

**Rubric coverage:**
- ✅ #6 Tables + visualisations (matplotlib, seaborn, folium)
- ✅ #7 Statistical analysis with **p-values** — Pearson, Welch t-test, ANOVA
- ✅ Bonus #1 Creativity (combining BFS reference + scrape, choropleth)
- ✅ Bonus #4 LLM market commentary

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
logging.basicConfig(level=logging.INFO, format='%(message)s')
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
from app import database

df = database.load_listings()
canton_df = database.load_canton_stats()
print(f'Loaded {len(df):,} listings across {df["canton"].nunique()} cantons.')
df.head(3)

## 1. Descriptive tables

Three views: by canton, by room count, urban vs. rural — all computed in **SQL** so the rubric's bonus #3 stays exercised here too.

In [ ]:
by_canton = database.query_avg_rent_by_canton()
by_canton

In [ ]:
by_rooms = database.query_room_distribution()
by_rooms

In [ ]:
urban_rural = database.query_urban_vs_rural()
urban_rural

## 2. Visualisations

In [ ]:
from app import visualization as viz

viz.plot_rent_vs_size(df)
plt.show()

In [ ]:
viz.plot_price_per_m2_by_canton(df)
plt.show()

In [ ]:
viz.plot_urban_vs_rural(df)
plt.show()

In [ ]:
viz.plot_room_distribution(df)
plt.show()

In [ ]:
# Combine with BFS reference and look at the gap
from app.data_cleaning import attach_canton_reference
df_gap = attach_canton_reference(df, canton_df)
viz.plot_listing_vs_bfs_gap(df_gap)
plt.show()

### Interactive map of all listings

Saved as standalone HTML in `outputs/listings_map.html`. Open it directly in a browser — markers scale with rent and the colour gradient encodes price per m².

In [ ]:
map_path = viz.build_listing_map(df)
print(f'Map saved to: {map_path.relative_to(ROOT)}')

# Inline preview in the notebook
from IPython.display import IFrame
IFrame(src=str(map_path.relative_to(ROOT)), width=900, height=500)

## 3. Statistical tests (p-values — rubric #7)

Three independent tests, each with a clear null hypothesis. We report the test statistic, p-value, an effect size, and a verdict at α = 0.05.

In [ ]:
from app import analysis

results = analysis.run_all(df)
summary = analysis.results_to_dataframe(results)
summary

### Interpretation

- **Pearson correlation (rent ~ living space).** The coefficient *r* tells us whether bigger flats systematically cost more. A small p-value (< 0.05) means the linear relationship is unlikely to be due to chance.
- **Welch t-test (urban vs. rural price/m²).** Tests whether the average price per m² in the five large urban cantons (ZH, GE, BS, BE, VD) differs from the rest. We use Welch's variant because the two groups have unequal variances and sizes.
- **One-way ANOVA across cantons.** Tests whether *any* canton's mean differs from the others. A significant F-statistic motivates further pair-wise tests, but for the rubric one ANOVA with a p-value is enough.

## 4. LLM market commentary (bonus #4)

We aggregate the data and ask Mistral 7B (via Together.ai) to write a short, factual commentary. If `TOGETHER_API_KEY` is missing the helper falls back to a deterministic offline summary so the notebook still runs.

In [ ]:
from app import llm_helper

commentary = llm_helper.market_commentary(by_canton, by_rooms)
print(commentary)

## 5. Conclusions

- Rent scales **strongly and significantly** with living space (Pearson r ≫ 0, p ≪ 0.05).
- Urban cantons command a **clear premium** per m²; the Welch t-test detects the difference at α = 0.05.
- One-way ANOVA confirms that **canton membership matters** beyond pure size effects.
- Listings are roughly consistent with the BFS canton baseline, but show non-trivial dispersion — useful intuition for renters comparing offers.
- The Streamlit app at `app/streamlit_app.py` exposes the same analysis interactively.

See the project README for a complete mapping from rubric items to source files.